# **3 - Validating**

## Importing Libraries

In [10]:
import psycopg2
import pandas as pd

## Connect to Database

In [11]:
connection = psycopg2.connect(
    
    host = 'localhost',
    port = '5432',
    database = 'demo',
    user = 'bryguan',
    password = 'qwertyuiop')

cursor = connection.cursor()

## Validating Data Types

### Customers

In [12]:
query = '''

SELECT sale_id::bigint, customer_id::bigint, first_name::text, last_name::text, street::text, city::text, state::text, zip::bigint
FROM customers
ORDER BY stage_id;

'''

connection.rollback()
display(pd.read_sql_query(query, connection))

,sale_id,customer_id,first_name,last_name,street,city,state,zip
0,5763728874,3728404,Darrelle,Dohrmann,46 Farwell Terrace,Oakland,CA,94609
1,5763729036,3729309,Moria,Greenwood,8803 Delaware Crossing,Berkeley,CA,94705
2,5763728904,3728508,Josiah,Hulett,6755 Melby Plaza,Oakland,CA,94612
3,5763728973,3728534,Gayle,MacGarrity,286 Onsgard Center,Berkeley,CA,94703
4,5763728757,3729188,Courtenay,Shirrell,75 West Park,Emeryville,CA,94608
...,...,...,...,...,...,...,...,...
92,5763728927,3728568,Valina,Spring,119 Sachtjen Junction,Berkeley,CA,94702
93,5763729096,3728990,Claire,Mebes,358 Oxford Road,Albany,CA,94706
94,5763729268,3728901,Freddy,Mumford,6 Transport Lane,Orinda,CA,94563
95,5763729237,3729019,Arv,Doret,2120 Mesta Circle,Emeryville,CA,94608


### Sales

In [13]:
query = '''

SELECT sale_id::bigint, sale_date::date, sub_total::int, tax::int, total::int
FROM sales;

'''

connection.rollback()
display(pd.read_sql_query(query, connection))

,sale_id,sale_date,sub_total,tax,total
0,5763728874,2020-10-03,12,0,12
1,5763729036,2020-10-03,72,0,72
2,5763728904,2020-10-03,24,0,24
3,5763728973,2020-10-03,96,0,96
4,5763728757,2020-10-03,108,0,108
...,...,...,...,...,...
92,5763728927,2020-10-03,72,0,72
93,5763729096,2020-10-03,48,0,48
94,5763729268,2020-10-03,84,0,84
95,5763729237,2020-10-03,60,0,60


### Stores

In [14]:
query = '''

SELECT sale_id::bigint, store_id::bigint, name::text, street::text, city::text, state::text, zip::bigint
FROM stores;

'''

connection.rollback()
display(pd.read_sql_query(query, connection))

,sale_id,store_id,name,street,city,state,zip
0,5763728874,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
1,5763729036,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
2,5763728904,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
3,5763728973,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
4,5763728757,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
...,...,...,...,...,...,...,...
92,5763728927,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
93,5763729096,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
94,5763729268,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705
95,5763729237,12573,Acme Gourmet Meals,3000 Telegraph Ave,Berkeley,CA,94705


### Products

In [15]:
query = '''

SELECT sale_id::bigint, item_id::integer, product_id::bigint, price::integer, quantity::integer
FROM products;

'''

connection.rollback()
display(pd.read_sql_query(query, connection))

,sale_id,item_id,product_id,price,quantity
0,5763728874,1,42314680,12,1
1,5763729036,1,42314677,12,1
2,5763729036,2,42314682,12,3
3,5763729036,3,42314684,12,2
4,5763728904,1,42314680,12,1
...,...,...,...,...,...
347,5763729237,2,42314678,12,2
348,5763729237,3,42314682,12,2
349,5763728673,1,42314677,12,2
350,5763728673,2,42314678,12,1


## Validating Data Logic

### Sales: Total = Sales: Sub-Total + Tax

In [16]:
query = '''

SELECT sale_id
FROM sales
WHERE total::numeric <> (sub_total::numeric + tax::numeric);

'''

connection.rollback()
display(pd.read_sql_query(query, connection))

,sale_id


### Sales: Total = Products: Price * Quantity

In [17]:
query = '''

SELECT sale_id
FROM sales
WHERE sales.total::numeric <> 
(SELECT SUM(products.price::numeric * products.quantity::numeric)
FROM products
WHERE sales.sale_id = products.sale_id);

'''

connection.rollback()
display(pd.read_sql_query(query, connection))

,sale_id


## Close Connection

In [18]:
connection.close()